In [1]:
# ==========================================
# PHASE 1: ENVIRONMENT SETUP & REAL DATA EDA
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier


In [3]:
# 1. Real Data Ingestion
# (Dataset source: Kaggle Credit Risk Dataset)
try:
    raw_data = pd.read_csv('credit_risk_dataset.csv')
except FileNotFoundError:
    raise FileNotFoundError("Please ensure 'credit_risk_dataset.csv' is saved in your local directory.")


In [4]:
# Standardizing raw columns to align with clean, intuitive variable names
# loan_int_rate serves as our core risk indicator (Higher Interest Rate = Higher Risk)
data = raw_data[[
    'loan_int_rate',
    'person_income',
    'person_age',
    'person_home_ownership',
    'loan_intent',
    'loan_status'
]].copy()
data.columns = ['interest_rate', 'annual_income', 'age', 'home_ownership', 'purpose', 'default']
print("--- Phase 1: Real Data Ingestion & Mapping Complete ---")


--- Phase 1: Real Data Ingestion & Mapping Complete ---


In [5]:
# 2. Target Assessment
target_counts = data['default'].value_counts()
target_pct = data['default'].value_counts(normalize=True) * 100
print(f"Real Target Distribution:\nClass 0 (Non-Default): {target_counts[0]} ({target_pct[0]:.2f}%)")
print(f"Class 1 (Default):     {target_counts[1]} ({target_pct[1]:.2f}%)\n")

# 3. Feature Isolation & Stratified Split
X = data.drop(columns=['default'])
y = data['default']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)


Real Target Distribution:
Class 0 (Non-Default): 25473 (78.18%)
Class 1 (Default):     7108 (21.82%)



In [6]:
# ==========================================
# PHASE 2: PREPROCESSING ARCHITECTURE
# ==========================================
# Automatically identifying numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Defining sub-pipelines (Handling real dataset missing values natively)
# SimpleImputer targets missing records, like unassigned interest rates
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Synthesizing into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print("--- Phase 2: Preprocessing Architecture Formed ---")



--- Phase 2: Preprocessing Architecture Formed ---


In [7]:
# ==========================================
# PHASE 3: MODEL EXECUTION
# ==========================================
# Calculating the exact algorithmic guardrail scaling parameter for real imbalanced data
negative_cases = y_train.value_counts()[0]
positive_cases = y_train.value_counts()[1]
calculated_scale_pos_weight = negative_cases / positive_cases

print(f"Calculated scale_pos_weight: {calculated_scale_pos_weight:.2f}")

# Constructing the master Pipeline
master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        scale_pos_weight=calculated_scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    ))
])

# Model Fit (Learns coefficients naturally from data)
master_pipeline.fit(X_train, y_train)
print("--- Phase 3: Master Pipeline Trained Successfully ---\n")


Calculated scale_pos_weight: 3.58
--- Phase 3: Master Pipeline Trained Successfully ---



In [8]:
# ==========================================
# PHASE 4: STATISTICAL EVALUATION (Standard 0.50 Threshold)
# ==========================================
y_pred_baseline = master_pipeline.predict(X_val)
y_probs = master_pipeline.predict_proba(X_val)[:, 1]

print("--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---")
print(classification_report(y_val, y_pred_baseline))
print(f"ROC-AUC Score: {roc_auc_score(y_val, y_probs):.4f}\n")

# ==========================================
# PHASE 5: FINANCIAL IMPACT ANALYSIS
# ==========================================
# Define enterprise cost matrix variables
COST_FP = 500   # Type I Error (Lost Opportunity Cost)
COST_FN = 5000  # Type II Error (Actual Default Principal Loss)

# TCO Calculation Function
def calculate_tco(y_true, y_pred_labels):
    cm = confusion_matrix(y_true, y_pred_labels)
    # Extract structural components of confusion matrix
    tn, fp, fn, tp = cm.ravel()
    tco = (fp * COST_FP) + (fn * COST_FN)
    return tco, tn, fp, fn, tp

# Evaluate default performance cost
baseline_tco, tn, fp, fn, tp = calculate_tco(y_val, y_pred_baseline)
print("--- Phase 5: Financial Impact Analysis ---")
print(f"Baseline TCO at 0.50 Threshold: ${baseline_tco:,}")
print(f"↳ False Positives (Type I): {fp} | False Negatives (Type II): {fn}\n")

# Threshold Optimization Analysis
thresholds = [0.50, 0.40, 0.30, 0.20, 0.15, 0.10, 0.05]
tco_results = []

print("Optimizing Decision Thresholds to Mitigate Risk:")
print("-" * 65)
print(f"{'Threshold':<12}{'False Pos (FP)':<16}{'False Neg (FN)':<16}{'Total TCO':<15}")
print("-" * 65)

for t in thresholds:
    # Overriding the default 0.50 threshold mapping manually
    y_pred_custom = (y_probs >= t).astype(int)
    cost, tn_c, fp_c, fn_c, tp_c = calculate_tco(y_val, y_pred_custom)
    tco_results.append((t, cost))
    print(f"{t:<12}{fp_c:<16}{fn_c:<16}${cost:<14,}")

print("-" * 65)
best_threshold, min_cost = min(tco_results, key=lambda item: item[1])
print(f"Optimal Business Decision Threshold discovered at: {best_threshold}")
print(f"Minimized Enterprise Financial Risk (TCO): ${min_cost:,}")



--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---
              precision    recall  f1-score   support

           0       0.93      0.82      0.87      5095
           1       0.55      0.77      0.64      1422

    accuracy                           0.81      6517
   macro avg       0.74      0.80      0.76      6517
weighted avg       0.84      0.81      0.82      6517

ROC-AUC Score: 0.8850

--- Phase 5: Financial Impact Analysis ---
Baseline TCO at 0.50 Threshold: $2,072,500
↳ False Positives (Type I): 915 | False Negatives (Type II): 323

Optimizing Decision Thresholds to Mitigate Risk:
-----------------------------------------------------------------
Threshold   False Pos (FP)  False Neg (FN)  Total TCO      
-----------------------------------------------------------------
0.5         915             323             $2,072,500     
0.4         1294            229             $1,792,000     
0.3         1745            139             $1,567,500     
0.2         23

Business-Driven Evaluation & Threshold Optimization Analysis

Executive Summary: The Conflict Between Statistics and Dollars

In enterprise risk management, a model that performs exceptionally well on paper can still lead to catastrophic financial losses if its deployment strategy ignores business realities. Standard machine learning evaluations rely on a default probability threshold of 0.50, assuming a symmetric world where a false alarm carries the same weight as a missed threat.

In credit risk modeling, however, mistakes are fundamentally asymmetric. Failing to detect a defaulting borrower costs an organization the entire lost principal of the loan, whereas turning away a qualified applicant merely results in a minor operational or lost-opportunity cost. By transitioning from a standard statistical assessment to a Business-Driven Evaluation, we can align our machine learning pipeline with the company's bottom line—ultimately restructuring our decision boundaries to minimize Total Cost of Ownership (TCO).

The Statistical Baseline: A Flawed Victory

Evaluating the credit risk model at the standard statistical baseline (0.50 threshold) reveals a highly competent predictive engine. The system achieves a robust ROC-AUC (Receiver Operating Characteristic-Area Under the Curve) Score of 0.8850, proving that the underlying features and the trained gradient-boosted classifier (XGBClassifier) possess excellent discriminatory power.

--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---

              precision    recall  f1-score   support
           0       0.93      0.82      0.87      5095
           1       0.55      0.77      0.64      1422

However, a deeper dive into the classification report exposes an operational vulnerability:

	Class 1 (Default) Precision (55%): Out of all borrowers flagged by the model as a default risk, nearly half are actually safe. This leads to a high volume of false alarms.

	Class 1 (Default) Recall (77%): The model captures 77% of actual defaults, which sounds strong statistically, but means that 23% of defaulting accounts are slipping through completely unnoticed.

When these percentages are processed through the corporate cost matrix—where a False Positive (FP) costs $500 in lost opportunity and a False Negative (FN) costs $5,000 in unrecoverable principal loss—the financial impact of the baseline strategy becomes starkly clear.

Baseline TCO at 0.50 Threshold: $2,072,500
 The Main Financial Cost

↳ False Positives (Type I): 915 | False Negatives (Type II): 323  
 The breakdown showing exactly WHY it costs that much

At a 0.50 default cutoff, the enterprise experiences 915 false alarms and misses 323 defaults, culminating in a heavy Baseline TCO of $2,072,500. Even though the statistical accuracy sits comfortably at 81%, the organization faces severe financial exposure because the missed defaults are ten times more expensive than the false alarms.

Threshold Optimization: Simulating Risk Mitigation

To protect the enterprise's capital, we must actively shift our risk tolerance. The threshold optimization loop serves as a financial simulation, systematically lowering the probability requirement for a borrower to be flagged as a default risk. By lowering the threshold from 0.50 down to 0.05, we purposefully trade an increase in cheap false alarms for a drastic reduction in catastrophic missed defaults.

Threshold   False Pos (FP)  False Neg (FN)  Total TCO      
-----------------------------------------------------------------
0.5         915             323             $2,072,500     
0.4         1294            229             $1,792,000     
0.3         1745            139             $1,567,500     
0.2         2301            82              $1,560,500     
0.15        2668            65              $1,659,000     
0.1         3099            43              $1,764,500     
0.05        3647            13              $1,888,500

As the cutoff falls, the financial dynamics shift across distinct phases:

	The Aggressive Cost-Cutting Phase (0.50→0.30): Dropping the threshold initially yields massive dividends. Even though False Positives climb from 915 to 1,745, the number of missed defaults is slashed by more than half (from 323 down to 139). This aggressive reduction drops the TCO by over $500,000.

	The Diminishing Returns Phase (0.20→0.05): Eventually, the strategy becomes overly sensitive. At a threshold of 0.05, the model captures almost every single default (only 13 are missed), but it flags an excessive 3,647 safe customers as risks. The massive operational overhead of these false alarms causes the TCO to climb back up to $1,888,500.

Conclusion: The Optimal Decision Boundary

By applying an automated optimization routine across our simulation ledger, the pipeline discovers the mathematical sweet spot for the enterprise's risk posture:

Optimal Business Decision Threshold discovered at: 0.2

Minimized Enterprise Financial Risk (TCO): $1,560,500

At the 0.20 probability threshold, the system balances both cost drivers perfectly. By telling the model to flag any applicant displaying even a modest 20% mathematical risk of default, the business accepts a higher volume of false alarms (2,301) in exchange for aggressively driving missed defaults down to just 82.

Financial Savings = Baseline TCO ($2,072,500) - Optimized TCO ($1,560,500)
                  = $512,000

Deploying the model with this business-driven threshold yields an immediate cost reduction of $512,000 for the validation cohort alone. This analysis proves that the ultimate goal of enterprise data science is not to chase abstract academic metrics like accuracy, but to use statistical intelligence to engineer data-driven safeguards that maximize bottom-line profitability.


